In [6]:

!pip -q install pymupdf sentence-transformers faiss-cpu transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 35.7 MB/s eta 0:00:00


In [7]:

from google.colab import files

uploaded = files.upload()

pdf_path = next(
    name for name in uploaded
    if name.lower().endswith(".pdf")
)

print("PDF uploaded:", pdf_path)


Saving sample_python_rag.pdf to sample_python_rag (1).pdf
PDF uploaded: sample_python_rag (1).pdf


In [8]:

import fitz

doc = fitz.open(pdf_path)

pages = []

for page_number, page in enumerate(doc, start=1):
    text = page.get_text()

    if text.strip():
        pages.append({
            "page": page_number,
            "text": text
        })

print("Total PDF pages:", len(doc))
print("Pages containing text:", len(pages))

if pages:
    print("\nFirst page preview:\n")
    print(pages[0]["text"][:1500])
else:
    print("No selectable text found. This PDF may need OCR.")


Total PDF pages: 1
Pages containing text: 1

First page preview:

Sample PDF for RAG Project
Introduction to Python
Python is a popular programming language created by Guido van Rossum and released in 1991.
Features of Python:
1. Easy to learn and use.
2. Supports object-oriented programming.
3. Open source.
4. Large community support.
5. Used in web development, data science, AI, and automation.
Creating Variables:
A variable is used to store data.
name = "Dwarak"
age = 21
Data Types:
int, float, string, list, tuple, dictionary.
Functions:
def greet():
    print("Hello World")
Machine Learning is a subset of Artificial Intelligence that enables systems to learn from data.



In [9]:

chunks = []

chunk_size = 800
overlap = 150

for page in pages:
    text = page["text"]
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))

        chunks.append({
            "page": page["page"],
            "text": text[start:end]
        })

        if end == len(text):
            break

        start += chunk_size - overlap

print("Total chunks created:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0]["text"] if chunks else "No chunks available")



Total chunks created: 1

First chunk:

Sample PDF for RAG Project
Introduction to Python
Python is a popular programming language created by Guido van Rossum and released in 1991.
Features of Python:
1. Easy to learn and use.
2. Supports object-oriented programming.
3. Open source.
4. Large community support.
5. Used in web development, data science, AI, and automation.
Creating Variables:
A variable is used to store data.
name = "Dwarak"
age = 21
Data Types:
int, float, string, list, tuple, dictionary.
Functions:
def greet():
    print("Hello World")
Machine Learning is a subset of Artificial Intelligence that enables systems to learn from data.



In [10]:

!pip install -q -U sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 7.2 MB/s eta 0:00:00


In [11]:

from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

print("New embedding model loaded successfully!")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

New embedding model loaded successfully!


In [12]:

texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
).astype("float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("FAISS index rebuilt!")
print("Total PDF chunks:", index.ntotal)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS index rebuilt!
Total PDF chunks: 1


In [13]:

def retrieve_context(question, top_k=4):
    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    k = min(top_k, index.ntotal)
    scores, indices = index.search(query_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:
            results.append({
                "page": chunks[int(idx)]["page"],
                "score": float(score),
                "text": chunks[int(idx)]["text"]
            })

    return results

print("JADX is ready to search your PDF!")


JADX is ready to search your PDF!


In [14]:

question = input("Ask a question about your PDF: ")

results = retrieve_context(question, top_k=4)

print("\n🔎 Relevant passages found:\n")

for i, result in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print(f"📄 Page: {result['page']}")
    print(f"🎯 Similarity: {result['score']:.4f}")
    print(result["text"][:800])
    print("-" * 50)


Ask a question about your PDF: what is the content of the pdf

🔎 Relevant passages found:


--- Result 1 ---
📄 Page: 1
🎯 Similarity: 0.5354
Sample PDF for RAG Project
Introduction to Python
Python is a popular programming language created by Guido van Rossum and released in 1991.
Features of Python:
1. Easy to learn and use.
2. Supports object-oriented programming.
3. Open source.
4. Large community support.
5. Used in web development, data science, AI, and automation.
Creating Variables:
A variable is used to store data.
name = "Dwarak"
age = 21
Data Types:
int, float, string, list, tuple, dictionary.
Functions:
def greet():
    print("Hello World")
Machine Learning is a subset of Artificial Intelligence that enables systems to learn from data.

--------------------------------------------------


In [15]:

!pip install -q gradio pymupdf sentence-transformers faiss-cpu transformers accelerate


In [17]:

import gradio as gr
import fitz
import faiss
import numpy as np
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

# -----------------------------
# 1. Load AI models
# -----------------------------
print("Loading JADX embedding model...")

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")

print("Loading JADX answer model...")

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

# -----------------------------
# 2. PDF processing
# -----------------------------
def process_pdf(pdf_file):
    if pdf_file is None:
        return None, "⚠️ Please upload a PDF first."

    path = pdf_file if isinstance(pdf_file, str) else pdf_file.name
    doc = fitz.open(path)

    chunks = []
    chunk_size = 800
    overlap = 150

    for page_no, page in enumerate(doc, start=1):
        text = page.get_text("text").strip()

        if not text:
            continue

        start = 0
        while start < len(text):
            end = min(start + chunk_size, len(text))

            chunks.append({
                "page": page_no,
                "text": text[start:end]
            })

            if end == len(text):
                break

            start += chunk_size - overlap

    if not chunks:
        return None, (
            "❌ No selectable text found. "
            "This PDF may require OCR."
        )

    texts = [c["text"] for c in chunks]

    vectors = embedder.encode(
        texts,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    ).astype("float32")

    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)

    state = {"chunks": chunks, "index": index}

    return state, (
        f"✅ PDF processed successfully!\n"
        f"Pages: {len(doc)} | Text chunks: {len(chunks)}"
    )

# -----------------------------
# 3. Retrieve relevant passages
# -----------------------------
def retrieve(question, state, top_k=4):
    query_vector = embedder.encode(
        [question],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    k = min(top_k, state["index"].ntotal)
    scores, ids = state["index"].search(query_vector, k)

    results = []

    for score, idx in zip(scores[0], ids[0]):
        if idx >= 0:
            results.append({
                **state["chunks"][int(idx)],
                "score": float(score)
            })

    return results

# -----------------------------
# 4. Generate answer with Qwen
# -----------------------------
def answer_question(question, history, state):
    if state is None:
        return "📄 Please upload and process a PDF first."

    if not question.strip():
        return "Please enter a question."

    results = retrieve(question, state)

    context = "\n\n".join(
        f"[Page {r['page']}]\n{r['text']}"
        for r in results
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are JADX, a helpful PDF assistant. "
                "Answer using only the supplied PDF context. "
                "If the answer is missing, say you could not "
                "find it in the PDF. Include source page numbers."
            )
        },
        {
            "role": "user",
            "content": f"PDF context:\n{context}\n\nQuestion: {question}"
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=6000
    ).to(llm.device)

    with torch.no_grad():
        output = llm.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    source_pages = sorted(set(r["page"] for r in results))

    sources = "\n".join(
        f"- Page {p}" for p in source_pages
    )

    return f"{answer}\n\n📚 **Source pages:**\n{sources}"

# -----------------------------
# 5. Gradio interface
# -----------------------------
CSS = """
.gradio-container {
    background: linear-gradient(135deg, #0b1020, #17152f, #101b30);
    color: #ffffff;
}
#title {
    text-align: center;
    color: #7cddff;
}
"""

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="purple"
    ),
    css=CSS,
    title="JADX PDF Reader AI"
) as demo:

    gr.Markdown(
        "# 📚 JADX PDF Reader AI\n"
        "### Upload a document. Ask questions. Discover answers.",
        elem_id="title"
    )

    state = gr.State(None)

    with gr.Row():
        with gr.Column(scale=1):
            pdf_input = gr.File(
                label="📄 Upload your PDF",
                file_types=[".pdf"],
                type="filepath"
            )

            process_btn = gr.Button(
                "⚡ Process PDF",
                variant="primary"
            )

            status = gr.Textbox(
                label="Document status",
                interactive=False
            )

            gr.Markdown(
                "**Powered by** BGE Embeddings · FAISS · Qwen"
            )

        with gr.Column(scale=2):
            chatbot = gr.Chatbot(
                label="💬 Chat with JADX",
                height=420
            )

            question = gr.Textbox(
                label="Ask your PDF",
                placeholder="What is this document about?"
            )

            ask_btn = gr.Button(
                "🚀 Ask JADX",
                variant="primary"
            )

            clear_btn = gr.Button("🗑️ Clear chat")

    process_btn.click(
        fn=process_pdf,
        inputs=pdf_input,
        outputs=[state, status]
    )

    def chat_fn(q, history, pdf_state):
        answer = answer_question(q, history, pdf_state)

        history = history or []
        history = history + [
            {"role": "user", "content": q},
            {"role": "assistant", "content": answer}
        ]

        return history, ""

    ask_btn.click(
        fn=chat_fn,
        inputs=[question, chatbot, state],
        outputs=[chatbot, question]
    )

    question.submit(
        fn=chat_fn,
        inputs=[question, chatbot, state],
        outputs=[chatbot, question]
    )

    clear_btn.click(
        fn=lambda: [],
        outputs=chatbot
    )

demo.launch(share=True)


Loading JADX embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading JADX answer model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/tmp/ipykernel_4700/2158386970.py:194: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://60904c4ce17fb9dd85.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
